# SBERT Sentence Analysis

## 1. Preparations

### 1.1 Read Data
This new data is obtained after some further anonymization, and by putting previously incorrectly split sentences together, and removed duplicates

In [ ]:
import pandas as pd
# read the sentence data 
df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx")
# check the head
df.head()

In [ ]:
# clean sentences as inputs
sentences = df["Clean_Sentence"].to_list()

In [ ]:
# print the size of the data
len(sentences)
# There are 41143 sentences 

### 1.2. Import the list of stopwords

In [ ]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/persistent/mijnidbcoachnlp/data/analysis_data/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl", "weken", "week", "dagen", "dag", "mg", "coach", "mijnibdcoach", "dr", "uur", "dgs"
]

dutch_stopwords.extend(extra_list)

### 1.4. Embed the lists of sentences with sentence-transformer multilingual-v1

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# load the embedding model
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")

# run the following for embeddings if no pre-saved embeddings can be loaded
#embeddings = embedding_model.encode(sentences, show_progress_bar=True)
# save the pre-calculated embeddings 
#np.save('/workspace/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy', embeddings)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
# load pre-saved embeddings if you have, otherwise calculate them using the commented-out codes
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")
embeddings = np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy")

In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Create a dictionary representation of the words in topics
tokenized_texts = [doc.split() for doc in sentences]  # Tokenizing by splitting on spaces
dictionary = Dictionary(tokenized_texts)

def get_top_words(topic_model):
    topics = topic_model.get_topics()
    top_words = [[word for word, _ in topic[:10]] for topic in topics.values() if topic]  # Top 10 words per topic
    return top_words

def calculate_c_v(topic_model):
    top_words = get_top_words(topic_model)
    coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')
    coherence_score = coherence_model.get_coherence()
    return coherence_score

In [ ]:
# disable parallelism to avoid some warnings
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 2. Fine-tune the ST model

In [ ]:
import random
import pickle
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# Precompute embeddings (this part depends on your chosen embedding model)
# Example: Assuming 'embeddings' is already computed and available
# embeddings = embedding_model.encode(sentences)

# Initialize BERTopic with the pre-fit vectorizer
topic_model = BERTopic()

# Define parameter ranges
range_n_neighbors = [10, 20, 30, 40, 50]
range_n_components = [2, 3, 4, 5, 6]
range_min_dist = [0.0, 0.05, 0.1]

# Number of random configurations to generate
num_configs = 3  # Adjust this as needed

# Generate random UMAP configurations
umap_configs = [
    {
        "n_neighbors": random.choice(range_n_neighbors),
        "n_components": random.choice(range_n_components),
        "min_dist": random.choice(range_min_dist),
    }
    for _ in range(num_configs)
]

# Define the ranges for HDBSCAN configurations
range_min_cluster_size = [10, 20, 30]  # Example values
range_min_samples = [10, 20, 30]  # Example values

# Generate random HDBSCAN configurations
hdbscan_configs = [
    {
        "min_cluster_size": random.choice(range_min_cluster_size),
        "min_samples": random.choice([x for x in range_min_samples if x <= random.choice(range_min_cluster_size)]),
    }
    for _ in range(num_configs)
]

# Initialize a dictionary to keep track of configurations and their associated coherence scores
tried_configs = {}

# Optionally, load previously tried configurations from a file (if running multiple times)
try:
    with open("tried_configs_st.pkl", "rb") as f:
        tried_configs = pickle.load(f)
except FileNotFoundError:
    pass  # If no previous file exists, we start fresh

# Variables to track the best model
best_topic_model = None
best_coherence_score = -float('inf')  # Initializing to a very low score
best_configure = None

# Set the outlier thresholds (upper and lower)
upper_outlier_threshold = 0.5  # Maximum allowed outlier percentage (5% of points)
lower_outlier_threshold = 0.2  # Minimum allowed outlier percentage (2% of points)

In [ ]:
# Fine-tuning loop
for umap_params in umap_configs:
    for hdbscan_params in hdbscan_configs:
        config_tuple = (frozenset(umap_params.items()), frozenset(hdbscan_params.items()))

        # Skip if the configuration has already been tried
        if config_tuple in tried_configs:
            print(f"Skipping already tried configuration: {umap_params}, {hdbscan_params}")
            continue

        # Initialize UMAP and HDBSCAN with current configurations
        umap_model = UMAP(**umap_params)
        hdbscan_model = HDBSCAN(**hdbscan_params)
        print(f"Fitting model for UMAP {umap_params} and HDBSCAN {hdbscan_params}")

        # Update BERTopic with the new UMAP and HDBSCAN models
        topic_model.umap_model = umap_model
        topic_model.hdbscan_model = hdbscan_model
        topic_model.vectorizer_model = CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b')
        topic_model.calculate_probabilities = False

        # Fit the model
        topics, probs = topic_model.fit_transform(sentences, embeddings)

        # Calculate coherence score
        current_coherence_score = calculate_c_v(topic_model)

        # Check if the model has at least 20 topics
        num_topics = len(topic_model.get_topics())
        if num_topics < 20:
            print(f"Skipping model with {num_topics} topics (less than 20).")
            tried_configs[config_tuple] = "skipped"
            continue

        # Check outlier percentage
        outlier_cluster_size = sum(1 for label in hdbscan_model.labels_ if label == -1)
        outlier_percentage = outlier_cluster_size / len(hdbscan_model.labels_)
        
        if outlier_percentage > upper_outlier_threshold:
            print(f"Skipping model with {outlier_percentage * 100}% outliers (above {upper_outlier_threshold * 100}%).")
            tried_configs[config_tuple] = "skipped"
            continue
        elif outlier_percentage < lower_outlier_threshold:
            print(f"Skipping model with {outlier_percentage * 100}% outliers (below {lower_outlier_threshold * 100}%).")
            tried_configs[config_tuple] = "skipped"
            continue
        
        # Store the valid configuration and score
        tried_configs[config_tuple] = current_coherence_score

        # Update best model
        if current_coherence_score > best_coherence_score:
            best_coherence_score = current_coherence_score
            best_topic_model = topic_model
            best_configure = (umap_params, hdbscan_params)

        print(f"Coherence score: {current_coherence_score}\n")

        # Save tried configurations after every iteration
        with open("tried_configs_st.pkl", "wb") as f:
            pickle.dump(tried_configs, f)


In [ ]:
# After the loop, print the best configurations and score
print("Best UMAP and HDBSCAN configurations based on coherence score:")
print(f"Best Coherence Score: {best_coherence_score}")
print(f"Best UMAP Params: {best_configure[0]}")
print(f"Best HDBSCAN Params: {best_configure[1]}")

In [ ]:
import pickle

# Load the file
with open("tried_configs_st.pkl", "rb") as f:
    tried_configs = pickle.load(f)

# Print the contents
for config, score in tried_configs.items():
    print(f"Config: {config}, Score: {score}")


In [ ]:
best_topic_model.save("/workspace/persistent/mijnidbcoachnlp/saved_models/best_cv_score_st_model", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [ ]:
calculate_c_v(best_topic_model)

In [ ]:
best_topic_model.get_topic_info()

In [ ]:
# check if reduce outliers now work
# Reduce outliers
topics = best_topic_model.topics_
new_topics = best_topic_model.reduce_outliers(sentences, topics)

In [ ]:
best_topic_model.update_topics(sentences, topics=new_topics, vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b'))

In [ ]:
# check topic_info after outlier reduction
best_topic_model.get_topic_info()

In [ ]:
best_topic_model.save("/workspace/persistent/mijnidbcoachnlp/saved_models/best_cv_score_st_model_ro", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [ ]:
calculate_c_v(best_topic_model)

##  3. Merge topics

In [ ]:
from bertopic import BERTopic
topic_model = BERTopic.load("/workspace/persistent/mijnidbcoachnlp/saved_models/best_cv_score_st_model_ro", embedding_model=embedding_model)

In [ ]:
topic_model.get_topic_info().head()

In [ ]:
# settings for plotly
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default = "browser" # option to open the plots in brower

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
#topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
# Reduce the topics with a similarity threshold < 0.5
threshold = 0.5 # Define the threshold for merging
filtered_hierarchical_topics = hierarchical_topics[hierarchical_topics["Distance"] <= threshold]

# Ensure the loop stops when no more topics can be merged
while not filtered_hierarchical_topics.empty:  

    list_topics_to_merge = filtered_hierarchical_topics["Topics"].to_list()

    # Merge topics
    topic_model.merge_topics(sentences, list_topics_to_merge)

    # Recompute hierarchical topics after merging
    hierarchical_topics = topic_model.hierarchical_topics(sentences)

    # Update the filtered topics for the next iteration
    filtered_hierarchical_topics = hierarchical_topics[hierarchical_topics["Distance"] <= threshold]


In [ ]:
topic_info  = topic_model.get_topic_info()
topic_info[topic_info["Name"].str.contains("thuistes")]

In [ ]:
doc_info = topic_model.get_document_info(sentences)
pd.set_option("display.max_colwidth", None)
doc_info[doc_info["Topic"] == 105]

In [ ]:
# settings for plotly
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default = "browser" # option to open the plots in brower

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
#topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
topics_to_merge_manually = [[190, 25],#food, appetite
[65, 3],# GI symptoms
[108, 54], #sleep
[171, 7, 89, 83, 57, 115, 60, 5, 93], #medicine/medication use
[92, 90, 2], #pharmacy and prescription
[193, 107, 173, 70], # telephone contact
[180, 87, 151], # vitamins and minerals
[197, 59], #inquire of test results
[80, 72], # advice
[113, 156, 18], # coronavirus & vaccine
[124, 142], #fever
[11, 130], # quetion/questionaire
[42, 94, 14, 183], #calprotectin home-test
[189, 20, 149, 162, 55, 102, 167, 208, 154, 53, 109, 187, 98, 78, 52, 101, 23, 22, 179] # sign-offs and greetings
]
topic_model.merge_topics(sentences, topics_to_merge_manually)


In [ ]:
topic_info = topic_model.get_topic_info()
topic_info[topic_info["Name"].str.contains("thuistes")]

In [ ]:
topic_names = topic_model.get_topic_info()["Name"].to_list()
topic_names

In [ ]:

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
#topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
doc_info = topic_model.get_document_info(sentences)
pd.set_option("display.max_colwidth", None)
doc_info[doc_info["Topic"] == 45]

In [ ]:
topics_to_merge_manually_2 = [[5, 131], # GI symptoms
[16, 8], # general symptoms
[34, 23], # diarrhea
[101, 3], # hospital/doctor visit
[20, 26, 13, 50], # appointment/session
[88, 95, 74], # calprotectin value
[18, 6], # home test
[45, 37], # stool sampling forms and papers
[29, 89, 134], # mail and address
[162, 67, 1], # sign-offs
[56, 104], # non-actionable 
]
topic_model.merge_topics(sentences, topics_to_merge_manually_2)


In [ ]:
topic_info = topic_model.get_topic_info()
topic_info[topic_info["Name"].str.contains("thuis")]

In [ ]:

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
#topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
topic_names = topic_model.get_topic_info()["Name"].to_list()
topic_names

In [ ]:
doc_info = topic_model.get_document_info(sentences)
pd.set_option("display.max_colwidth", 200)
doc_info[doc_info["Topic"] == 57]

In [ ]:
# large merge
topics_to_merge_manually_3 = [[145, 147, 9, 123], # hoor graag
[25, 68], # want to know
[33, 34], # results inquiry
[102, 42], # weet
[92, 55], # thought process
[143, 1],
[78, 100], # operation
[18, 63, 59, 39, 44, 4, 57] # appointment and contact
]
topic_model.merge_topics(sentences, topics_to_merge_manually_3)


In [ ]:
topic_model.get_topic_info()[:10]

In [ ]:
# large merge
topics_to_merge_manually_4 = [[106, 78, 7, 8, 39, 24, 10, 66], # symptoms and complaints
[0, 6], # stool and blood test
[21, 15, 63], # calprotectin home test
[14, 2], #contact and appointment
]
topic_model.merge_topics(sentences, topics_to_merge_manually_4)


In [ ]:

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
#topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:

topic_info = topic_model.get_topic_info()

In [ ]:
topic_info = topic_model.get_topic_info()
topic_names = topic_info["Name"].to_list()
topic_names

In [ ]:
labels = [
    "A",  # -1_verstandigst_trant_aangenaam_zenuwachtig
    "M",  # 0_bloed_prikken_ontlasting_test
    "M",  # 1_pijn_buikpijn_toilet_last
    "A",  # 2_afspraak_telefonisch_contact_bellen
    "A",  # 3_bedankt_dank_reactie_snelle
    "M",  # 4_medicatie_gebruik_tabletten_medicijnen
    "M",  # 5_huisarts_arts_dokter_mdl
    "A",  # 6_recept_apotheek_nieuw_sturen
    "M",  # 7_bekend_calprotectine_uitslag_waarde
    "A",  # 8_hoor_graag_horen_verneem
    "A",  # 9_vraag_vragen_vraagje_vragenlijst
    "M",  # 10_voel_gevoel_voelt_voelen
    "M",  # 11_corona_vaccinatie_vaccin_positief
    "A",  # 12_formulieren_formulier_potje_papieren
    "A",  # 13_mail_brief_adres_bericht
    "A",  # 14_weten_laat_vertellen_laten
    "A",  # 15_hoop_wacht_wachten_hopelijk
    "M",  # 16_goed_gaat_ging_algemeen
    "M",  # 17_eten_eet_eetlust_drinken
    "M",  # 18_advies_adviseren_overleggen_graag
    "M",  # 19_infuus_komende_echo_vanmiddag
    "A",  # 20_weet_ermee_goed_houdt
    "M",  # 21_crohn_ziekte_verklaring_humira
    "A",  # 22_prima_helemaal_allemaal_zsm
    "M",  # 23_beter_gebeurd_gaat_stuk
    "M",  # 24_ijzer_vitamine_ijzerinfuus_foliumzuur
    "A",  # 25_maanden_maand_vragenlijst_maandelijks
    "A",  # 26_mogelijk_mogelijkheid_manier_snel
    "M",  # 27_stoppen_rekening_cortiment_doorgaan
    "A",  # 28_idee_denk_dacht_meld
    "A", #29
    "A",  # 30_geregeld_opsturen_gekeken_doorgeven
    "M",  # 31_humira_hyrimoz_gespoten_spuiten
    "A",  # 32_morgen_vanmorgen_morgenvroeg_ophalen
    "A",  # 33_jaar_vorig_jaren_leeftijd
    "M",  # 34_controle_ziekenhuis_check_periodieke
    "A",  # 35_vernomen_gebeld_helaas_extra
    "M",  # 36_dermatoloog_huid_dermatologie_gezicht
    "A",  # 37_volgende_mailen_hoeft_spuiten
    "A",  # 38_mee_hieraan_vreemd_valt
    "M",  # 39_antibiotica_kuur_antibioticakuur_bacterie
    "M",  # 40_operatie_chirurg_ingepland_gevolgen
    "A",  # 41_gelukkig_blij_gelukt_tevreden
    "A",  # 42_vakantie_vertrek_terug_reizen
    "M",  # 43_onderzoek_akkoord_melden_meedoen
    "A",  # 44_ibd_ibdcoach_sessie_vragen
    "M",  # 45_duurt_ruim_natuurlijk_lang
    "M",  # 46_koorts_temperatuur_verhoging_koude
    "A",  # 47_gaan_besproken_betekenen_houd
    "M",  # 48_mri_scan_dunne_uitslag
    "M",  # 49_zorgen_begin_ongerust_echt
    "M",  # 50_zwangerschap_zwanger_gynaecoloog_medicijngebruik
    "A",  # 51_fijne_feestdagen_jaarwisseling_gewenst
    "A",  # 52_plannen_inplannen_termijn_afspraak
    "A",  # 53_doorgeven_geven_volledig_oplossing
    "M",  # 54_keuze_opvlamming_bang_gal
    "A",  # 55_gegaan_mis_verkeerd_fout
    "A",  # 56_ontvangen_heden_moviprep_bevestiging
    "A",  # 57_zorginstelling_apotheek_ziekenhuis_naam
    "M",  # 58_coloscopie_endoscopie_darmonderzoek_verhuisd
    "A",  # 59_sorry_excuses_late_lastig
    "A",  # 60_zie_vlak_zien_verkeerde
    "A",  # 61_thuistesten_verwijzing_nieuwe_versturen
    "M",  # 62_dosering_combinatie_dosis_laag
    "A",  # 63_resultaten_hiervan_resultaat_vernemen
    "A",  # 64_bedoeling_teveel_doorgegeven_vergoed
    "A",  # 65_wilde_reden_dringend_denkt
    "M",  # 66_veranderd_infusen_helft_interval
    "A",  # 67_app_downloaden_telefoon_ondersteund
    "A",  # 68_voorraad_doosje_doos_stuks
    "A",  # 69_bespreken_spreken_persoonlijk_praten
    "A",  # 70_beste_hey_betreft_aandoen
    "M",  # 71_spuit_doorsturen_metamucil_injecties
    "M",  # 72_gewicht_kilo_afgevallen_weeg
    "M",  # 73_lactose_gestopt_beangstigend_gedaald
    "A",  # 74_nachten_duidelijk_kort_bestaat
    "A",  # 75_foto_bijgevoegd_bestand_bijgaand
    "A",  # 76_probleem_toekomst_echt_problemen
    "M",  # 77_stress_invloed_druk_rol
    "A",  # 78_zekerheid_navragen_vragenlijsten_checken
    "M",  # 79_pillen_systeem_azathiopirine_heen
    "A",  # 80_gehoord_gehoor_heden_kapper
    "M",  # 81_ontsteking_galstenen_idd_aanvallen
    "A",  # 82_vergeten_herinnering_herinneren_geheugen
    "M",  # 83_gestart_entyvio_gevaccineerd_verhuizen
    "A",  # 84_vandaar_gevraagd_griep_stuur
    "M",  # 85_ophalen_gang_fistel_bezig
    "M",  # 86_maagonderzoek_kloppen_vervolgafspraak_begrepen
    "A",  # 87_druppels_has_are_kale
    "M",  # 88_rechts_linker_steken_inspanning
    "A",  # 89_hiervoor_situatie_ten_eraan
    "A",  # 90_voldoende_hopende_hiermee_hoop
    "A",  # 91_gaten_houden_hou_gehouden
    "A",  # 92_helpen_kastje_hiermee_muur
    "A",  # 93_optie_alternatief_momenten_mogelijkheden
    "A",  # 94_vermelden_originele_hoor_biologische
    "A",  # 95_aanhouden_verstandig_rest_vervolg
    "A",  # 96_bekeken_lukken_betekend_geeft
    "M",  # 97_biologicals_biological_bureau_wratten
    "M",  # 98_risico_risicogroep_val_loop
    "M",  # 99_urine_kweek_gecontroleerd_urinekweek
    "A",  # 100_fax_faxnummer_vriendelijk_tel
    "M",  # 101_energie_energielevel_moe_stukken
    "A",  # 102_annuleren_meewerken_uitsluitsel_tijdstip
    "A",  # 103_toestemming_toestemmingsformulier_geef_bijlage
    "A",  # 104_wekelijks_nou_volgen_geplande
    "A",  # 105_verzekering_aanvraag_gegevens_levensverzekering
    "A",  # 106_duren_verbetering_langs_werken
    "M",  # 107_werkt_aangeef_traject_darmkrampen
    "A",  # 108_wensen_allereerst_gezond_wens
    "M",  # 109_bijwerking_bijwerkingen_infliximab_innemen
    "A",  # 110_eerstvolgende_aanstaande_groot_terecht
    "M",  # 111_problemen_lange_hierdoor_persen
    "A",  # 112_fysiek_opeens_heden_straks
    "A",  # 113_team_ibd_binnenkort_matig
    "A",  # 114_gezondheidscheck_periodieke_invullen_vullen
    "A",  # 115_gelezen_laboratorium_afgesproken_buikgriep
    "M",  # 116_tandarts_tandvlees_kaakchirurg_mond
    "A",  # 117_betekent_vervolgstappen_stappen_the
    "M",  # 118_aangepast_baat_beclometason_tabletjes
    "A",  # 119_tegemoet_zie_reactie_graag
    "A",  # 120_doorgestuurd_picoprep_gestuurd_zorginstelling
    "A",  # 121_geboortedatum_geboorte_geb_geboren
]
len(labels)

In [ ]:
topic_summaries = [
    "outlier",  # -1_verstandigst_trant_aangenaam_zenuwachtig
    "blood/stool test",  # 0_bloed_prikken_ontlasting_test
    "symptoms",  # 1_pijn_buikpijn_toilet_last
    "appointment scheduling",  # 2_afspraak_telefonisch_contact_bellen
    "other",  # 3_bedankt_dank_reactie_snelle
    "medication use",  # 4_medicatie_gebruik_tabletten_medicijnen
    "doctor/hospital contact",  # 5_huisarts_arts_dokter_mdl
    "prescription refill",  # 6_recept_apotheek_nieuw_sturen
    "calprotectin results",  # 7_bekend_calprotectine_uitslag_waarde
    "other",  # 8_hoor_graag_horen_verneem
    "questions/inquiries",  # 9_vraag_vragen_vraagje_vragenlijst
    "feelings",  # 10_voel_gevoel_voelt_voelen
    "COVID-19 vaccination",  # 11_corona_vaccinatie_vaccin_positief
    "forms/documents",  # 12_formulieren_formulier_potje_papieren
    "email/mail communication",  # 13_mail_brief_adres_bericht
    "other",  # 14_weten_laat_vertellen_laten
    "other",  # 15_hoop_wacht_wachten_hopelijk
    "general health update",  # 16_goed_gaat_ging_algemeen
    "eating/drinking",  # 17_eten_eet_eetlust_drinken
    "advice",  # 18_advies_adviseren_overleggen_graag
    "infusion/echo appointment",  # 19_infuus_komende_echo_vanmiddag
    "other",  # 20_weet_ermee_goed_houdt
    "Crohn's disease/Humira",  # 21_crohn_ziekte_verklaring_humira
    "other",  # 22_prima_helemaal_allemaal_zsm
    "general health update",  # 23_beter_gebeurd_gaat_stup
    "iron/vitamin",  # 24_ijzer_vitamine_ijzerinfuus_foliumzuur
    "monthly updates",  # 25_maanden_maand_vragenlijst_maandelijks
    "possibilities/options",  # 26_mogelijk_mogelijkheid_manier_snel
    "medication continuation",  # 27_stoppen_rekening_cortiment_doorgaan
    "thoughts/ideas",  # 28_idee_denk_dacht_meld
    "morning_plans", #29
    "follow-up",  # 30_geregeld_opsturen_gekeken_doorgeven
    "Humira/Hyrimoz injections",  # 31_humira_hyrimoz_gespoten_spuiten
    "other",  # 32_morgen_vanmorgen_morgenvroeg_ophalen
    "age/years",  # 33_jaar_vorig_jaren_leeftijd
    "hospital check-up",  # 34_controle_ziekenhuis_check_periodieke
    "follow-up",  # 35_vernomen_gebeld_helaas_extra
    "dermatology/skin",  # 36_dermatoloog_huid_dermatologie_gezicht
    "next steps",  # 37_volgende_mailen_hoeft_spuiten
    "confusion/concerns",  # 38_mee_hieraan_vreemd_valt
    "antibiotics",  # 39_antibiotica_kuur_antibioticakuur_bacterie
    "surgery/operation",  # 40_operatie_chirurg_ingepland_gevolgen
    "happiness/satisfaction",  # 41_gelukkig_blij_gelukt_tevreden
    "travel/vacation",  # 42_vakantie_vertrek_terug_reizen
    "research participation",  # 43_onderzoek_akkoord_melden_meedoen
    "other",  # 44_ibd_ibdcoach_sessie_vragen
    "waiting time",  # 45_duurt_ruim_natuurlijk_lang
    "fever/temperature",  # 46_koorts_temperatuur_verhoging_koude
    "discussions",  # 47_gaan_besproken_betekenen_houd
    "MRI scan",  # 48_mri_scan_dunne_uitslag
    "worries/concerns",  # 49_zorgen_begin_ongerust_echt
    "pregnancy/medication",  # 50_zwangerschap_zwanger_gynaecoloog_medicijngebruik
    "other",  # 51_fijne_feestdagen_jaarwisseling_gewenst
    "planning/scheduling",  # 52_plannen_inplannen_termijn_afspraak
    "information sharing",  # 53_doorgeven_geven_volledig_oplossing
    "flare-up concerns",  # 54_keuze_opvlamming_bang_gal
    "mistakes/issues",  # 55_gegaan_mis_verkeerd_fout
    "receipt confirmation",  # 56_ontvangen_heden_moviprep_bevestiging
    "healthcare facility",  # 57_zorginstelling_apotheek_ziekenhuis_naam
    "colonoscopy/endoscopy",  # 58_coloscopie_endoscopie_darmonderzoek_verhuisd
    "apologies",  # 59_sorry_excuses_late_lastig
    "other",  # 60_zie_vlak_zien_verkeerde
    "requests for supplies",  # 61_thuistesten_verwijzing_nieuwe_versturen
    "dosage adjustments",  # 62_dosering_combinatie_dosis_laag
    "results discussion",  # 63_resultaten_hiervan_resultaat_vernemen
    "intentions/overpayment",  # 64_bedoeling_teveel_doorgegeven_vergoed
    "urgent requests",  # 65_wilde_reden_dringend_denkt
    "infusion changes",  # 66_veranderd_infusen_helft_interval
    "mobile app",  # 67_app_downloaden_telefoon_ondersteund
    "medication stock",  # 68_voorraad_doosje_doos_stuks
    "personal discussions",  # 69_bespreken_spreken_persoonlijk_praten
    "greetings",  # 70_beste_hey_betreft_aandoen
    "injections",  # 71_spuit_doorsturen_metamucil_injecties
    "weight changes",  # 72_gewicht_kilo_afgevallen_weeg
    "lactose intolerance",  # 73_lactose_gestopt_beangstigend_gedaald
    "sleep issues",  # 74_nachten_duidelijk_kort_bestaat
    "attached files",  # 75_foto_bijgevoegd_bestand_bijgaand
    "future problems",  # 76_probleem_toekomst_echt_problemen
    "stress impact",  # 77_stress_invloed_druk_rol
    "confirmation/questions",  # 78_zekerheid_navragen_vragenlijsten_checken
    "medication system",  # 79_pillen_systeem_azathiopirine_heen
    "general updates",  # 80_gehoord_gehoor_heden_kapper
    "inflammation/gallstones",  # 81_ontsteking_galstenen_idd_aanvallen
    "forgetfulness/memory",  # 82_vergeten_herinnering_herinneren_geheugen
    "Entyvio start/vaccination",  # 83_gestart_entyvio_gevaccineerd_verhuizen
    "inquiries/flu",  # 84_vandaar_gevraagd_griep_stuur
    "fistula issues",  # 85_ophalen_gang_fistel_bezig
    "stomach examination",  # 86_maagonderzoek_kloppen_vervolgafspraak_begrepen
    "drops/medication",  # 87_druppels_has_are_kale
    "pain locations",  # 88_rechts_linker_steken_inspanning
    "other",  # 89_hiervoor_situatie_ten_eraan
    "other",  # 90_voldoende_hopende_hiermee_hoop
    "other",  # 91_gaten_houden_hou_gehouden
    "assistance/help",  # 92_helpen_kastje_hiermee_muur
    "options/alternatives",  # 93_optie_alternatief_momenten_mogelijkheden
    "biological medication",  # 94_vermelden_originele_hoor_biologische
    "continuation/rest",  # 95_aanhouden_verstandig_rest_vervolg
    "other",  # 96_bekeken_lukken_betekend_geeft
    "biologicals treatment",  # 97_biologicals_biological_bureau_wratten
    "risk factors",  # 98_risico_risicogroep_val_loop
    "urine test",  # 99_urine_kweek_gecontroleerd_urinekweek
    "fax communication",  # 100_fax_faxnummer_vriendelijk_tel
    "energy levels",  # 101_energie_energielevel_moe_stukken
    "cancellation/cooperation",  # 102_annuleren_meewerken_uitsluitsel_tijdstip
    "consent forms",  # 103_toestemming_toestemmingsformulier_geef_bijlage
    "weekly updates",  # 104_wekelijks_nou_volgen_geplande
    "insurance matters",  # 105_verzekering_aanvraag_gegevens_levensverzekering
    "duration/improvement",  # 106_duren_verbetering_langs_werken
    "treatment effectiveness",  # 107_werkt_aangeef_traject_darmkrampen
    "other",  # 108_wensen_allereerst_gezond_wens
    "medication side effects",  # 109_bijwerking_bijwerkingen_infliximab_innemen
    "next appointment",  # 110_eerstvolgende_aanstaande_groot_terecht
    "long-term issues",  # 111_problemen_lange_hierdoor_persen
    "physical changes",  # 112_fysiek_opeens_heden_straks
    "other",  # 113_team_ibd_binnenkort_matig
    "health check",  # 114_gezondheidscheck_periodieke_invullen_vullen
    "lab results",  # 115_gelezen_laboratorium_afgesproken_buikgriep
    "dental issues",  # 116_tandarts_tandvlees_kaakchirurg_mond
    "next steps",  # 117_betekent_vervolgstappen_stappen_the
    "medication adjustment",  # 118_aangepast_baat_beclometason_tabletjes
    "other",  # 119_tegemoet_zie_reactie_graag
    "forwarded documents",  # 120_doorgestuurd_picoprep_gestuurd_zorginstelling
    "birthdate",  # 121_geboortedatum_geboorte_geb_geboren
]

len(topic_summaries)

In [ ]:
topic_info[topic_info["Name"].str.contains("operatie")]

In [ ]:
topic_info["Label"] = labels
topic_info["Summary"] = topic_summaries
labeled_topic_info = topic_info

In [ ]:
non_informative_topics = labeled_topic_info["Topic"][labeled_topic_info["Summary"] == "other"].to_list()
non_informative_topics

In [ ]:
meaningful_topic_info = labeled_topic_info[labeled_topic_info["Summary"] != "other"][1:]

In [ ]:
medication_topics = meaningful_topic_info[meaningful_topic_info["Summary"].str.contains("medication|injection|dosage|treatment|vitamin|antibio")]
remaining_topics = meaningful_topic_info[~meaningful_topic_info["Summary"].str.contains("medication|injection|dosage|treatment|vitamin|antibio")]
medication_topics


In [ ]:
# medication topics to merge
[94, 97], #biologicals
[62, 118], #dose adjustment
[68, 79] # medication stock
[31, 71] # injections

In [ ]:
health_update_topics = remaining_topics[remaining_topics["Summary"].str.contains("update|follow-up|improvement|check")]
remaining_topics = remaining_topics[~remaining_topics["Summary"].str.contains("update|follow-up|improvement|check")]
health_update_topics

In [ ]:
health_update_topics["Topic"].to_list()
# all merge together

In [ ]:
test_topics = remaining_topics[remaining_topics["Summary"].str.contains("test|lab|improvement|calpro")]
remaining_topics = remaining_topics[~remaining_topics["Summary"].str.contains("test|lab|improvement|calpro")]
test_topics
#[0, 115]

In [ ]:
contact_topics = remaining_topics[remaining_topics["Summary"].str.contains("appointment|call|telephone|fax|email|mail")]
contact_topics
#[2, 110]

In [ ]:
remaining_topics = remaining_topics[~remaining_topics["Summary"].str.contains("appointment|call|telephone|fax|email|mail")]
remaining_topics

In [ ]:
remaining_topics[49:]

In [ ]:

# visualize the hierarchy and manually merge some topics
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
topics_to_merge_5 = [[94, 97], #biologicals
[62, 118], #dose adjustment
[68, 79], # medication stock
[31, 71], # injections
[16, 23, 25, 30, 34, 35, 80, 104, 106, 114], # health updates
[2, 110, 39, 52, 29], # appointment
[0, 115], #bloodtest
[37, 49] # worries/concerns
]

topic_model.merge_topics(sentences, topics_to_merge_5)

In [ ]:
topic_info_after_merge = topic_model.get_topic_info()

In [ ]:
topic_info_after_merge["Name"].to_list()

In [ ]:
new_topic_summaries = [
    "outlier",  # -1_verstandigst_trant_aangenaam_zenuwachtig
    "blood/stool test",  # 0_bloed_prikken_ontlasting_test
    "appointment scheduling",  # 1_afspraak_telefonisch_contact_bellen
    "symptoms",  # 2_pijn_buikpijn_toilet_last
    "other",  # 3_bedankt_dank_reactie_snelle
    "medication use",  # 4_medicatie_gebruik_tabletten_medicijnen
    "health update",  # 5_maanden_beter_maand_goed
    "doctor/hospital visit",  # 6_huisarts_arts_dokter_mdl
    "prescription refill",  # 7_recept_apotheek_nieuw_sturen
    "calprotectin results",  # 8_bekend_uitslag_calprotectine_waarde
    "other",  # 9_hoor_graag_horen_verneem
    "questions/inquiries",  # 10_vraag_vragen_vraagje_vragenlijst
    "feelings",  # 11_voel_gevoel_voelt_voelen
    "COVID-19 vaccination",  # 12_corona_vaccinatie_vaccin_positief
    "forms/documents",  # 13_formulieren_formulier_potje_papieren
    "email/mail communication",  # 14_mail_brief_adres_bericht
    "other",  # 15_weten_laat_vertellen_laten
    "other",  # 16_hoop_wacht_wachten_hopelijk
    "worries",  # 17_zorgen_volgende_begin_mailen
    "eating/drinking habits",  # 18_eten_eet_eetlust_drinken
    "Humira injections",  # 19_humira_spuit_doorsturen_injecties
    "seeking advice",  # 20_advies_adviseren_graag_overleggen
    "infusion/echo follow-up",  # 21_infuus_komende_echo_terug
    "other",  # 22_weet_ermee_goed_houdt
    "Crohn's disease/Humira",  # 23_crohn_ziekte_verklaring_humira
    "other",  # 24_prima_helemaal_allemaal_zsm
    "iron/vitamin supplements",  # 25_ijzer_vitamine_ijzerinfuus_foliumzuur
    "possibilities/options",  # 26_mogelijk_mogelijkheid_manier_snel
    "medication continuation",  # 27_stoppen_rekening_cortiment_doorgaan
    "thoughts/ideas",  # 28_idee_denk_dacht_meld
    "other",  # 30_morgen_vanmorgen_morgenvroeg_ophalen
    "age/years",  # 31_jaar_vorig_jaren_leeftijd
    "dermatology/skin issues",  # 32_dermatoloog_huid_dermatologie_gezicht
    "confusion/concerns",  # 33_mee_hieraan_vreemd_valt
    "surgery/operation",  # 34_operatie_chirurg_ingepland_gevolgen
    "happiness/satisfaction",  # 35_gelukkig_blij_gelukt_tevreden
    "travel/vacation",  # 36_vakantie_vertrek_terug_reizen
    "research participation",  # 37_onderzoek_akkoord_melden_meedoen
    "medication stock/system",  # 38_pillen_voorraad_systeem_azathiopirine
    "waiting time",  # 39_duurt_ruim_natuurlijk_lang
    "other",  # 40_ibd_ibdcoach_sessie_vragen
    "fever/temperature",  # 41_koorts_temperatuur_verhoging_koude
    "discussions/meaning",  # 42_gaan_besproken_betekenen_houd
    "MRI scan results",  # 43_mri_scan_dunne_uitslag
    "pregnancy/medication",  # 44_zwangerschap_zwanger_gynaecoloog_medicijngebruik
    "other",  # 45_fijne_feestdagen_jaarwisseling_gewenst
    "information/communication",  # 46_doorgeven_geven_volledig_oplossing
    "dosage adjustments",  # 47_dosering_combinatie_dosis_laag
    "flare-up",  # 48_keuze_opvlamming_bang_gal
    "mistakes/issues",  # 49_gegaan_mis_verkeerd_fout
    "biological medication",  # 50_biologicals_vermelden_originele_biological
    "receipt confirmation",  # 51_ontvangen_heden_moviprep_bevestiging
    "healthcare facility",  # 52_zorginstelling_apotheek_ziekenhuis_naam
    "colonoscopy/darmonderzoek",  # 53_coloscopie_endoscopie_darmonderzoek_verhuisd
    "other",  # 54_zie_vlak_zien_verkeerde
    "other",  # 55_sorry_excuses_late_lastig
    "home tests/referrals",  # 56_thuistesten_verwijzing_nieuwe_versturen
    "results discussion",  # 57_resultaten_hiervan_resultaat_vernemen
    "goal",  # 58_bedoeling_teveel_doorgegeven_vergoed
    "urgent requests",  # 59_wilde_reden_dringend_denkt
    "infusion changes",  # 60_veranderd_infusen_helft_interval
    "app download",  # 61_app_downloaden_telefoon_ondersteund
    "discussions",  # 62_bespreken_spreken_persoonlijk_praten
    "other",  # 63_beste_hey_betreft_bellen
    "weight changes",  # 64_gewicht_kilo_afgevallen_weeg
    "lactose intolerance",  # 65_lactose_gestopt_komt_beangstigend
    "attached files",  # 66_foto_bijgevoegd_bestand_bijgaand
    "sleep issues",  # 67_nachten_duidelijk_kort_bestaat
    "future problems",  # 68_probleem_toekomst_echt_problemen
    "stress impact",  # 69_stress_invloed_druk_rol
    "confirmation/questions",  # 70_zekerheid_navragen_checken_vragenlijsten
    "inflammation/gallstones",  # 71_ontsteking_galstenen_idd_aanvallen
    "forgetfulness",  # 72_vergeten_herinnering_herinneren_geheugen
    "Entyvio start/vaccination",  # 73_gestart_entyvio_gevaccineerd_verhuizen
    "inquires",  # 74_vandaar_gevraagd_griep_stuur
    "stomach examination",  # 75_maagonderzoek_kloppen_vervolgafspraak_begrepen
    "fistula issues",  # 76_ophalen_gang_fistel_bezig
    "drops/medication",  # 77_druppels_has_are_kale
    "pain locations",  # 78_rechts_linker_steken_inspanning
    "other",  # 79_hiervoor_situatie_ten_eraan
    "other",  # 80_voldoende_hopende_hiermee_hoop
    "other",  # 81_gaten_houden_hou_gehouden
    "assistance",  # 82_helpen_hiermee_kastje_muur
    "options/alternatives",  # 83_optie_alternatief_momenten_mogelijkheden
    "continuation/rest",  # 84_aanhouden_verstandig_rest_vervolg
    "other",  # 85_bekeken_lukken_betekend_geeft
    "risk factors",  # 86_risico_risicogroep_val_loop
    "urine test",  # 87_urine_kweek_gecontroleerd_urinekweek
    "fax communication",  # 88_fax_faxnummer_vriendelijk_tel
    "energy levels",  # 89_energie_energielevel_moe_stukken
    "cancellation/cooperation",  # 90_annuleren_meewerken_uitsluitsel_tijdstip
    "consent forms",  # 91_toestemming_toestemmingsformulier_geef_bijlage
    "insurance matters",  # 92_verzekering_aanvraag_gegevens_levensverzekering
    "treatment effectiveness",  # 93_werkt_aangeef_traject_darmkrampen
    "other",  # 94_wensen_allereerst_gezond_wens
    "medication side effects",  # 95_bijwerking_bijwerkingen_infliximab_innemen
    "long-term issues",  # 96_problemen_lange_hierdoor_persen
    "physical changes",  # 97_fysiek_opeens_heden_straks
    "other",  # 98_team_ibd_binnenkort_matig
    "dental issues",  # 99_tandarts_tandvlees_kaakchirurg_mond
    "next steps",  # 100_betekent_vervolgstappen_stappen_the
    "other",  # 101_tegemoet_zie_reactie_graag
    "forwarded documents",  # 102_doorgestuurd_picoprep_gestuurd_zorginstelling
    "birthdate",  # 103_geboortedatum_geboorte_geb_geboren
]

In [ ]:
new_labels = [
    "A",  # -1_verstandigst_trant_aangenaam_zenuwachtig - 
    "A",  # 0_bloed_prikken_ontlasting_test - 
    "A",  # 1_afspraak_telefonisch_contact_bellen - 
    "M",  # 2_pijn_buikpijn_toilet_last - 
    "A",  # 3_bedankt_dank_reactie_snelle - 
    "M",  # 4_medicatie_gebruik_tabletten_medicijnen -
    "A",  # 5_maanden_beter_maand_goed - 
    "M",  # 6_huisarts_arts_dokter_mdl - 
    "A",  # 7_recept_apotheek_nieuw_sturen - 
    "A",  # 8_bekend_uitslag_calprotectine_waarde - 
    "A",  # 9_hoor_graag_horen_verneem - 
    "A",  # 10_vraag_vragen_vraagje_vragenlijst  -
    "M",  # 11_voel_gevoel_voelt_voelen -
    "M",  # 12_corona_vaccinatie_vaccin_positief - 
    "A",  # 13_formulieren_formulier_potje_papieren - 
    "A",  # 14_mail_brief_adres_bericht - 
    "A",  # 15_weten_laat_vertellen_laten -
    "A",  # 16_hoop_wacht_wachten_hopelijk - 
    "M",  # 17_zorgen_volgende_begin_mailen - 
    "M",  # 18_eten_eet_eetlust_drinken - 
    "M",  # 19_humira_spuit_doorsturen_injecties - 
    "M",  # 20_advies_adviseren_graag_overleggen - 
    "A",  # 21_infuus_komende_echo_terug - # labeled "A" as it is likely talking about appointment of infusion & echo
    "A",  # 22_weet_ermee_goed_houdt -
    "M",  # 23_crohn_ziekte_verklaring_humira - 
    "A",  # 24_prima_helemaal_allemaal_zsm -
    "M",  # 25_ijzer_vitamine_ijzerinfuus_foliumzuur - 
    "A",  # 26_mogelijk_mogelijkheid_manier_snel -
    "M",  # 27_stoppen_rekening_cortiment_doorgaan -
    "A",  # 28_idee_denk_dacht_meld - 
    "A",  # 29_morgen_vanmorgen_morgenvroeg_ophalen -
    "M",  # 30_jaar_vorig_jaren_leeftijd - 
    "M",  # 31_dermatoloog_huid_dermatologie_gezicht - 
    "M",  # 32_mee_hieraan_vreemd_valt -
    "M",  # 33_operatie_chirurg_ingepland_gevolgen - 
    "M",  # 34_gelukkig_blij_gelukt_tevreden - 
    "A",  # 35_vakantie_vertrek_terug_reizen -
    "A",  # 36_onderzoek_akkoord_melden_meedoen - 
    "M",  # 37_pillen_voorraad_systeem_azathiopirine - 
    "A",  # 38_duurt_ruim_natuurlijk_lang - 
    "M",  # 39_ibd_ibdcoach_sessie_vragen - 
    "M",  # 40_koorts_temperatuur_verhoging_koude - 
    "M",  # 41_gaan_besproken_betekenen_houd -
    "M",  # 42_mri_scan_dunne_uitslag - 
    "M",  # 43_zwangerschap_zwanger_gynaecoloog_medicijngebruik - 
    "A",  # 44_fijne_feestdagen_jaarwisseling_gewenst - 
    "A",  # 45_doorgeven_geven_volledig_oplossing - 
    "M",  # 46_dosering_combinatie_dosis_laag -
    "M",  # 47_keuze_opvlamming_bang_gal - 
    "A",  # 48_gegaan_mis_verkeerd_fout - 
    "M",  # 49_biologicals_vermelden_originele_biological
    "A",  # 50_ontvangen_heden_moviprep_bevestiging - 
    "A",  # 51_zorginstelling_apotheek_ziekenhuis_naam-
    "M",  # 52_coloscopie_endoscopie_darmonderzoek_verhuisd
    "A",  # 53_zie_vlak_zien_verkeerde -
    "A",  # 54_sorry_excuses_late_lastig - 
    "A",  # 55_thuistesten_verwijzing_nieuwe_versturen - 
    "A",  # 57_resultaten_hiervan_resultaat_vernemen  -
    "M",  # 58_bedoeling_teveel_doorgegeven_vergoed - 
    "A",  # 59_wilde_reden_dringend_denkt -
    "M",  # 60_veranderd_infusen_helft_interval - 
    "A",  # 61_app_downloaden_telefoon_ondersteund -
    "A",  # 62_bespreken_spreken_persoonlijk_praten - 
    "A",  # 63_beste_hey_betreft_bellen -
    "M",  # 64_gewicht_kilo_afgevallen_weeg - 
    "M",  # 65_lactose_gestopt_komt_beangstigend - 
    "A",  # 66_foto_bijgevoegd_bestand_bijgaand - 
    "M",  # 67_nachten_duidelijk_kort_bestaat -
    "M",  # 68_probleem_toekomst_echt_problemen - 
    "M",  # 69_stress_invloed_druk_rol - 
    "M",  # 70_zekerheid_navragen_checken_vragenlijsten - 
    "M",  # 71_ontsteking_galstenen_idd_aanvallen - 
    "A",  # 72_vergeten_herinnering_herinneren_geheugen - 
    "M",  # 73_gestart_entyvio_gevaccineerd_verhuizen -
    "M",  # 74_vandaar_gevraagd_griep_stuur - 
    "A",  # 75_maagonderzoek_kloppen_vervolgafspraak_begrepen - 
    "M",  # 76_ophalen_gang_fistel_bezig - 
    "M",  # 77_druppels_has_are_kale - 
    "M",  # 78_rechts_linker_steken_inspanning - 
    "A",  # 79_hiervoor_situatie_ten_eraan - 
    "A",  # 80_voldoende_hopende_hiermee_hoop - 
    "A",  # 81_gaten_houden_hou_gehouden -
    "M",  # 82_helpen_hiermee_kastje_muur -
    "A",  # 83_optie_alternatief_momenten_mogelijkheden
    "A",  # 84_aanhouden_verstandig_rest_vervolg
    "A",  # 85_bekeken_lukken_betekend_geeft
    "M",  # 86_risico_risicogroep_val_loop
    "M",  # 87_urine_kweek_gecontroleerd_urinekweek
    "A",  # 88_fax_faxnummer_vriendelijk_tel
    "M",  # 89_energie_energielevel_moe_stukken
    "A",  # 90_annuleren_meewerken_uitsluitsel_tijdstip
    "A",  # 91_toestemming_toestemmingsformulier_geef_bijlage
    "A",  # 92_verzekering_aanvraag_gegevens_levensverzekering
    "M",  # 93_werkt_aangeef_traject_darmkrampen
    "A",  # 94_wensen_allereerst_gezond_wens
    "M",  # 95_bijwerking_bijwerkingen_infliximab_innemen
    "M",  # 96_problemen_lange_hierdoor_persen
    "A",  # 97_fysiek_opeens_heden_straks
    "A",  # 97_team_ibd_binnenkort_matig
    "M",  # 98_tandarts_tandvlees_kaakchirurg_mond
    "M",  # 99_betekent_vervolgstappen_stappen_the
    "A",  # 100_tegemoet_zie_reactie_graag
    "A",  # 101_doorgestuurd_picoprep_gestuurd_zorginstelling
    "A",  # 102_geboortedatum_geboorte_geb_geboren
]

In [ ]:
len(topic_info_after_merge)

In [ ]:

labeled_topic_info_merged = topic_info_after_merge
labeled_topic_info_merged.head()

In [ ]:
final_meaningful_topic_info = labeled_topic_info_merged[labeled_topic_info_merged["Summary"] != "other"][1:]
final_meaningful_topic_info

In [ ]:
final_meaningful_topic_info.to_excel("/workspace/persistent/mijnidbcoachnlp/data/results/final_meaningful_topics.xlsx", index=False)

In [ ]:
topic_model.save("/workspace/persistent/mijnidbcoachnlp/saved_models/final_st_model", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

## 4. Visualize the meaningful topics

In [ ]:
import pandas as pd
final_meaningful_topic_info = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/results/final_meaningful_topics.xlsx")

In [ ]:
%matplotlib inline
%matplotlib widget

In [ ]:
df = final_meaningful_topic_info[["Count", "Summary", "Label"]]


In [ ]:
df = final_meaningful_topic_info[["Count", "Summary", "Label"]]
pivot_df = df.pivot(index='Summary', columns='Label', values='Count').fillna(0)
import pandas as pd
import matplotlib.pyplot as plt

# Add a total count column, which is the sum of counts for each label across both categories
pivot_df['Total'] = pivot_df['A'] + pivot_df['M']

# Sort by the 'Total' column in descending order (highest to lowest)
pivot_df = pivot_df.sort_values(by='Total', ascending=False).head(20)

# Create a figure and axes
fig, ax = plt.subplots(figsize=(10, 6))

# Plot Category M on the left side of the plot (negative values for left-side bars)
bars_m = ax.barh(pivot_df.index, pivot_df['M'], color='#A8D5BA', label='Medical', height=0.6)  # Softer green

# Plot Category A on the right side of the plot (positive values for right-side bars)
bars_a = ax.barh(pivot_df.index, pivot_df['A'], color='#A7C7E7', label='Non-Medical', height=0.6)  # Softer blue

# Add labels and title
ax.set_xlabel('Count')
ax.set_title('Sentence-level topic distribution')
ax.legend()

# Invert y-axis so the highest values are at the top
ax.invert_yaxis()

plt.savefig("/workspace/persistent/mijnidbcoachnlp/data/results/figure1.png")

# Show the plot
plt.tight_layout()
plt.show()




In [ ]:
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt


topic_info = pd.DataFrame(df)

# Create a dictionary where keys are the topic labels and values are their counts
word_freq = dict(zip(topic_info['Summary'], topic_info['Count']))

# Generate the word cloud using the dictionary
wordcloud = WordCloud(width=1200, height=400, background_color='white').generate_from_frequencies(word_freq)

# Plot the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.savefig("/workspace/persistent/mijnidbcoachnlp/data/results/figure2.png")

plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Pivot the data to have 'A' and 'M' in separate columns
pivot_df = df.pivot(index='Summary', columns='Label', values='Count').fillna(0)

# Select the top 10 topics for each category, sorted in descending order
top_10_M = pivot_df[['M']].sort_values(by='M', ascending=False).head(10)
top_10_A = pivot_df[['A']].sort_values(by='A', ascending=False).head(10)

# Create figure and axes for two plots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot Top 10 Medical (M)
axes[0].barh(top_10_M.index[::-1], top_10_M['M'][::-1], color='#A8D5BA', label='Medical')  # Reverse order
axes[0].set_xlabel('Count')
axes[0].set_title('Top 10 Medical Topics', fontsize=20)
axes[0].tick_params(axis='y', labelsize=20)  # Set y-axis tick label font size
axes[0].tick_params(axis='x', labelsize=14)  # Set y-axis tick label font size


# Plot Top 10 Administrative (A)
axes[1].barh(top_10_A.index[::-1], top_10_A['A'][::-1], color='#A7C7E7', label='Non_Medical')  # Reverse order
axes[1].set_xlabel('Count')
axes[1].set_title('Top 10 Non-Medical Topics', fontsize=20)
axes[1].tick_params(axis='y', labelsize=20)  # Set y-axis tick label font size
axes[1].tick_params(axis='x', labelsize=14)  # Set y-axis tick label font size


# Adjust layout and save the figure
plt.tight_layout()
plt.savefig("/workspace/persistent/mijnidbcoachnlp/data/results/plot_sep.png")


In [ ]:
df_m = final_meaningful_topic_info[final_meaningful_topic_info["Label"] == "M"]
df_a = final_meaningful_topic_info[final_meaningful_topic_info["Label"] == "a"]

In [ ]:
annotated_df_new = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/new_annotated_df_lg.xlsx")
annotated_df_new.head()

In [ ]:
# now we need to move the labels of the topic_info to doc_info
# Now perform the merge 
annotated_doc_info = pd.merge( # merging the small annotated set with the document info
    annotated_df_new,
    doc_info[['Sentence_ID', 'Topic', 'Top_n_words']], # notice that we only choose these 3 columns of the doc info df
    on='Sentence_ID',
    how='left'
)

annotated_doc_info.head()

In [ ]:
# now merge the above dataframe with the labeled topic info frame
annotated_doc_info = pd.merge(
    annotated_doc_info,
    topic_info[['Topic', 'Label']],
    on='Topic',
    how='left'
)
annotated_doc_info.head()

In [ ]:
annotated_doc_info[500:600]

In [ ]:
accuracy = (annotated_doc_info["Label"] == annotated_doc_info["Manual_Category"]).mean()

print(f"Accuracy: {accuracy:.2%}")

In [ ]:
# Accuracy for "M"
m_accuracy = (annotated_doc_info[annotated_doc_info["Label"] == "M"]["Manual_Category"] == "M").mean()

# Accuracy for "A"
a_accuracy = (annotated_doc_info[annotated_doc_info["Label"] == "A"]["Manual_Category"] == "A").mean()

print(f"Accuracy for M: {m_accuracy:.2%}")
print(f"Accuracy for A: {a_accuracy:.2%}")


In [ ]:
topic_info

In [ ]:

# Example DataFrame (replace with your actual data)

df = topic_info[topic_info["Label"] == "M"]
df = df[["Count", "Name", "Label"]][1:]
df.head()

 ## 5. Calculate accuracy

In [ ]:
from bertopic import BERTopic
topic_model = BERTopic.load("/workspace/persistent/mijnidbcoachnlp/saved_models/final_st_model", embedding_model=embedding_model)

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info.tail(10)

In [ ]:
topic_info["Category"] = new_labels
topic_info["Label"] = new_topic_summaries
topic_info.head()

In [ ]:
topic_info[topic_info["Topic"] == 52]

In [ ]:
annotated_df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/new_annotated_df_lg.xlsx")
print(len(annotated_df))
annotated_df.head()

In [ ]:
doc_info = topic_model.get_document_info(sentences)
doc_info["Sentence_ID"] = doc_info.index + 1
doc_info.head()

In [ ]:
# Now perform the merge 
annotated_df_with_topic = pd.merge( # merging the small annotated set with the document info
    annotated_df,
    doc_info[['Sentence_ID', 'Topic', 'Top_n_words']], # notice that we only choose these 3 columns of the doc info df
    on='Sentence_ID',
    how='left'
)

In [ ]:
annotated_df_with_topic.head()

In [ ]:
annotated_doc = pd.merge(
    annotated_df_with_topic,
    topic_info[['Topic', 'Label', 'Category']],
    on='Topic',
    how='left'
)

In [ ]:
annotated_doc.head()

In [ ]:
# Pring the average accuracy
accuracy_avg = (annotated_doc['Manual_Category'] == annotated_doc['Category']).sum() / len(annotated_doc)
print(f"Average Accuracy for Manual_Label: {accuracy_avg * 100:.2f}%")

In [ ]:
sentence_df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx")
sentence_df

In [ ]:
doc_info_labeled = pd.merge(
    doc_info[["Document", "Topic", "Name", "Representation", "Top_n_words", "Sentence_ID"]],
    topic_info[['Topic', 'Label', 'Category']],
    on="Topic",
    how="left"
)

In [ ]:
doc_info_labeled.head(10)

In [ ]:
doc_info_labeled_translated = pd.merge(
    doc_info_labeled,
    sentence_df[["Message_ID", "Sentence_ID", "Translated_Sentence"]],
    on="Sentence_ID",
    how="left")

In [ ]:
doc_info_labeled_translated

In [ ]:
message_category_df = doc_info_labeled_translated[["Category", "Message_ID"]]
message_category_df

In [ ]:
import pandas as pd
# Count occurrences of "A" and "M" for each Message ID
result = message_category_df.groupby("Message_ID")["Category"].value_counts().unstack(fill_value=0)

# Rename columns for clarity
result.columns.name = None  # Remove category name
result = result.rename(columns={"A": "Count_A", "M": "Count_M"})


In [ ]:
result = result.reset_index()
result[10:30]

In [ ]:

sentence_df[sentence_df["Message_ID"] == 35]

In [ ]:
result.to_excel("/workspace/persistent/mijnidbcoachnlp/data/results/message_categories.xlsx", index=False)

In [ ]:
nr_no_med_messages = len(result[result["Count_M"] == 0])
proportion = nr_no_med_messages / len(result)
print(f"Number of non-medical messages: {nr_no_med_messages}")
print(f"Proportion of non-medical messages: {proportion * 100:.2f}%")